In [20]:
import pandas as pd

# Đọc dữ liệu mới từ file Excel
data = pd.read_excel("my_reviews.xlsx")
print(f"Kích thước dữ liệu: {data.shape}")
print("\nCác cột trong data:")
print(data.columns.tolist())
print("\nMột số dòng đầu tiên:")
data.head()

Kích thước dữ liệu: (19806, 7)

Các cột trong data:
['reviewer_id', 'timestamp', 'review_content', 'company_name', 'company_type', 'company_size', 'company_address']

Một số dòng đầu tiên:


,reviewer_id,timestamp,review_content,company_name,company_type,company_size,company_address
0,369e5a,3 giờ trước,Công ty có nhiều dự án và lương thưởng đủ sống...,Việt Mỹ Ssu (15),NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...
1,f5083e,1 năm trước,Công ty có nhiều dự án shopee rất đáng để quan...,Việt Mỹ Ssu (15),NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...
2,c87f2d,1 năm trước,Mình vừa bị công ty quỵt tiền càng ngày càng x...,Việt Mỹ Ssu (15),NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...
3,6e7059,1 năm trước,Mình vừa bị công ty quỵt tiền càng ngày càng x...,Việt Mỹ Ssu (15),NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...
4,d6bae5,1 năm trước,"Thử việc ko hợp đồng, nhà có việc gia đình chụ...",Việt Mỹ Ssu (15),NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...


In [21]:
# Kiểm tra dữ liệu
print("Thông tin về dữ liệu:")
print(f"Tổng số reviews: {len(data)}")
print(f"Số công ty: {data['company_name'].nunique()}")
print(f"\nKiểm tra giá trị null:")
print(data.isnull().sum())
print(f"\nMột số review mẫu:")
data[["company_name", "review_content"]].head()

Thông tin về dữ liệu:
Tổng số reviews: 19806
Số công ty: 369

Kiểm tra giá trị null:
reviewer_id            0
timestamp              0
review_content         4
company_name           0
company_type       19806
company_size       19806
company_address     1452
dtype: int64

Một số review mẫu:


,company_name,review_content
0,Việt Mỹ Ssu (15),Công ty có nhiều dự án và lương thưởng đủ sống...
1,Việt Mỹ Ssu (15),Công ty có nhiều dự án shopee rất đáng để quan...
2,Việt Mỹ Ssu (15),Mình vừa bị công ty quỵt tiền càng ngày càng x...
3,Việt Mỹ Ssu (15),Mình vừa bị công ty quỵt tiền càng ngày càng x...
4,Việt Mỹ Ssu (15),"Thử việc ko hợp đồng, nhà có việc gia đình chụ..."


In [22]:
# Xóa phần số lượng review trong ngoặc sau tên công ty, vd: "Việt Mỹ Ssu (15)" -> "Việt Mỹ Ssu"
import re

data["company_name"] = data["company_name"].apply(
    lambda x: re.sub(r'\s*\(\d+\)\s*$', '', x).strip() if isinstance(x, str) else x
)

print("Đã xóa phần ngoặc sau tên công ty!")
print(f"Ví dụ một số tên công ty sau khi xử lý:")
print(data["company_name"].head(10).tolist())


Đã xóa phần ngoặc sau tên công ty!
Ví dụ một số tên công ty sau khi xử lý:
['Việt Mỹ Ssu', 'Việt Mỹ Ssu', 'Việt Mỹ Ssu', 'Việt Mỹ Ssu', 'Việt Mỹ Ssu', 'Việt Mỹ Ssu', 'Việt Mỹ Ssu', 'Việt Mỹ Ssu', 'Việt Mỹ Ssu', 'Việt Mỹ Ssu']


In [23]:
import re
import emoji
import html
def clean_review(text):
    if not isinstance(text, str) or not text:
        return ""

    text = html.unescape(text)

    # BƯỚC 1: Xử lý các biến thể của xuống dòng để split chính xác
    text = text.replace('\\n', '\n')

    # BƯỚC 2: Xóa toàn bộ emoji Unicode và emoticon text
    text = emoji.replace_emoji(text, replace=' ')
    text = re.sub(r':[Dd]+\b', ' ', text)   # :D :d :DD
    text = re.sub(r':\)+', ' ', text)        # :) :)) :)))
    text = re.sub(r':\(+', ' ', text)        # :( :((
    text = re.sub(r'>[-_.]<', ' ', text)     # >_< >.< >-<
    text = re.sub(r'\(\^[-_]\^\)', ' ', text)
    text = re.sub(r'\^[._-]?\^', ' ', text)
    text = re.sub(r'\(\^o?\^\)', ' ', text)
    # Bắt emoticon dạng :v :p :P :3 :x :D :d :) :( và biến thể
    text = re.sub(r':[a-zA-Z0-9\)\(]+', ' ', text)
    # BƯỚC 3: Tách dữ liệu thành từng dòng (List of lines)
    lines = text.split('\n')
    cleaned_lines = []

    # BƯỚC 4: Duyệt qua từng dòng để xóa sạch dấu gạch, số, sao...
    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Xóa # ở đầu dòng
        line = re.sub(r'^#\s*', '', line)
        
        # Xóa các ký tự đầu dòng không mong muốn
        line = re.sub(r'^[ \t\-\*\+\•\d\.\)\(\[\]]+', '', line)
        
        # Xóa các ký tự đặc biệt cuối dòng như ^, *, #
        line = re.sub(r'[\^\*#]+\s*$', '', line)
        
        # Xóa các dấu chấm liên tiếp
        line = re.sub(r'\.{2,}', '', line)
        
        # Xóa dấu phẩy cuối câu
        line = re.sub(r',\s*$', '', line)
        
        # Dọn dẹp khoảng trắng
        line = re.sub(r'\s+', ' ', line).strip()

        if line:
            # Thêm dấu chấm nếu chưa có dấu câu cuối
            if not re.search(r'[.!?]$', line):
                line += '.'
            cleaned_lines.append(line)

    # BƯỚC 5: Loại bỏ duplicate sentences (giữ thứ tự, không phân biệt hoa thường)
    seen = set()
    unique_lines = []
    for line in cleaned_lines:
        key = line.lower().strip()
        if key not in seen:
            seen.add(key)
            unique_lines.append(line)
    
    result = ' '.join(unique_lines)
    result = re.sub(r'\.+', '.', result)
    result = re.sub(r'\s\.', '.', result)
    result = re.sub(r'\s+', ' ', result)
    
    return result.strip()


def mask_company_name_in_text(text, company_name):
    """
    Mask tên công ty thành [COMPANY] trong text.
    Bao gồm: tên đầy đủ, các từ viết hoa trong tên (VMO, FPT, ...),
    và viết tắt ghép chữ cái đầu của từng từ.
    """
    if not isinstance(text, str) or not text or not isinstance(company_name, str):
        return text

    patterns_to_mask = []

    # 1. Tên đầy đủ công ty
    patterns_to_mask.append(re.escape(company_name))

    words = company_name.split()

    # 2. Các từ viết hoa hoàn toàn trong tên (vd: VMO, FPT, KPMG) — có thể là viết tắt
    for word in words:
        clean_word = re.sub(r'[^\w]', '', word)
        if len(clean_word) >= 2 and clean_word.isupper():
            patterns_to_mask.append(re.escape(clean_word))

    # 3. Viết tắt ghép chữ cái đầu của mỗi từ (vd: "VMO Holdings" -> "VH")
    initials = ''.join(w[0].upper() for w in words if w and w[0].isalpha())
    if len(initials) >= 2:
        patterns_to_mask.append(re.escape(initials))

    # Ghép tất cả pattern và thực hiện mask (ưu tiên pattern dài hơn trước)
    patterns_to_mask = sorted(set(patterns_to_mask), key=len, reverse=True)
    combined_pattern = r'\b(' + '|'.join(patterns_to_mask) + r')\b'
    result = re.sub(combined_pattern, 'Company A B C', text, flags=re.IGNORECASE)

    # Dọn dẹp khoảng trắng thừa
    result = re.sub(r'\s+', ' ', result).strip()

    return result


In [24]:
# Làm sạch nội dung review
print("Đang làm sạch nội dung review...")
data["review_content_cleaned"] = data["review_content"].astype("str").apply(clean_review)
print("Đã làm sạch xong!")
print(f"\nĐộ dài trung bình review sau khi clean: {data['review_content_cleaned'].str.len().mean():.0f} ký tự")
data[["review_content", "review_content_cleaned"]].head()

Đang làm sạch nội dung review...
Đã làm sạch xong!

Độ dài trung bình review sau khi clean: 293 ký tự


,review_content,review_content_cleaned
0,Công ty có nhiều dự án và lương thưởng đủ sống...,Công ty có nhiều dự án và lương thưởng đủ sống...
1,Công ty có nhiều dự án shopee rất đáng để quan...,Công ty có nhiều dự án shopee rất đáng để quan...
2,Mình vừa bị công ty quỵt tiền càng ngày càng x...,Mình vừa bị công ty quỵt tiền càng ngày càng x...
3,Mình vừa bị công ty quỵt tiền càng ngày càng x...,Mình vừa bị công ty quỵt tiền càng ngày càng x...
4,"Thử việc ko hợp đồng, nhà có việc gia đình chụ...","Thử việc ko hợp đồng, nhà có việc gia đình chụ..."


In [25]:
# Mask tên công ty trong review content
print("Đang mask tên công ty từ review...")

# Hàm xử lý mask cho từng dòng
def mask_company_in_review(row):
    review = row["review_content_cleaned"]
    company = row["company_name"]
    # Lấy tên công ty (bỏ phần số lượng review trong ngoặc nếu có)
    import re
    company_clean = re.sub(r'\s*\(\d+\)$', '', company) if isinstance(company, str) else company
    return mask_company_name_in_text(review, company_clean)

data["review_content_masked"] = data.apply(mask_company_in_review, axis=1)

print("Đã mask tên công ty xong!")
print("\nKiểm tra một vài dòng sau khi mask:")
data[["company_name", "review_content_masked"]].head(10)

Đang mask tên công ty từ review...
Đã mask tên công ty xong!

Kiểm tra một vài dòng sau khi mask:


,company_name,review_content_masked
0,Việt Mỹ Ssu,Công ty có nhiều dự án và lương thưởng đủ sống...
1,Việt Mỹ Ssu,Công ty có nhiều dự án shopee rất đáng để quan...
2,Việt Mỹ Ssu,Mình vừa bị công ty quỵt tiền càng ngày càng x...
3,Việt Mỹ Ssu,Mình vừa bị công ty quỵt tiền càng ngày càng x...
4,Việt Mỹ Ssu,"Thử việc ko hợp đồng, nhà có việc gia đình chụ..."
5,Việt Mỹ Ssu,"Môi trường vui vẻ nè, cô lao công cũng thân th..."
6,Việt Mỹ Ssu,"Rất tệ, train không lương nha. đừng nộp CV vào..."
7,Việt Mỹ Ssu,"T*** né cty này đi, nhất là cái dự án KOL, côn..."
8,Việt Mỹ Ssu,"Công ty lương thì thấp, mà leader hay đòi hỏi ..."
9,Việt Mỹ Ssu,Ứ** viên nộp CV vô thì tụi nó đăng CV của ứng ...


In [26]:
# Xóa các review trùng lặp (giữ lần đầu)
print("\nĐang xóa duplicate reviews...")
before_count = len(data)

# Xóa duplicate dựa trên review_content_masked
data_cleaned = data.drop_duplicates(subset=["review_content_masked"], keep="first")

# Loại bỏ các review rỗng sau khi làm sạch
data_cleaned = data_cleaned[data_cleaned["review_content_masked"].str.strip() != ""]

after_count = len(data_cleaned)
print(f"Đã xóa {before_count - after_count} reviews trùng lặp hoặc rỗng")
print(f"Số reviews còn lại: {after_count}")

data_cleaned.head(10)


Đang xóa duplicate reviews...
Đã xóa 1042 reviews trùng lặp hoặc rỗng
Số reviews còn lại: 18764


,reviewer_id,timestamp,review_content,company_name,company_type,company_size,company_address,review_content_cleaned,review_content_masked
0,369e5a,3 giờ trước,Công ty có nhiều dự án và lương thưởng đủ sống...,Việt Mỹ Ssu,NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...,Công ty có nhiều dự án và lương thưởng đủ sống...,Công ty có nhiều dự án và lương thưởng đủ sống...
1,f5083e,1 năm trước,Công ty có nhiều dự án shopee rất đáng để quan...,Việt Mỹ Ssu,NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...,Công ty có nhiều dự án shopee rất đáng để quan...,Công ty có nhiều dự án shopee rất đáng để quan...
2,c87f2d,1 năm trước,Mình vừa bị công ty quỵt tiền càng ngày càng x...,Việt Mỹ Ssu,NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...,Mình vừa bị công ty quỵt tiền càng ngày càng x...,Mình vừa bị công ty quỵt tiền càng ngày càng x...
4,d6bae5,1 năm trước,"Thử việc ko hợp đồng, nhà có việc gia đình chụ...",Việt Mỹ Ssu,NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...,"Thử việc ko hợp đồng, nhà có việc gia đình chụ...","Thử việc ko hợp đồng, nhà có việc gia đình chụ..."
5,3bb9c6,1 năm trước,"Môi trường vui vẻ nè, cô lao công cũng thân th...",Việt Mỹ Ssu,NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...,"Môi trường vui vẻ nè, cô lao công cũng thân th...","Môi trường vui vẻ nè, cô lao công cũng thân th..."
6,388b3f,2 năm trước,"Rất tệ, train không lương nha. đừng nộp CV vào...",Việt Mỹ Ssu,NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...,"Rất tệ, train không lương nha. đừng nộp CV vào...","Rất tệ, train không lương nha. đừng nộp CV vào..."
7,fe5494,2 năm trước,"T*** né cty này đi, nhất là cái dự án KOL, côn...",Việt Mỹ Ssu,NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...,"T*** né cty này đi, nhất là cái dự án KOL, côn...","T*** né cty này đi, nhất là cái dự án KOL, côn..."
8,f5a7f3,2 năm trước,"Công ty lương thì thấp, mà leader hay đòi hỏi ...",Việt Mỹ Ssu,NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...,"Công ty lương thì thấp, mà leader hay đòi hỏi ...","Công ty lương thì thấp, mà leader hay đòi hỏi ..."
9,2a4ad4,2 năm trước,Ứ** viên nộp CV vô thì tụi nó đăng CV của ứng ...,Việt Mỹ Ssu,NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...,Ứ** viên nộp CV vô thì tụi nó đăng CV của ứng ...,Ứ** viên nộp CV vô thì tụi nó đăng CV của ứng ...
10,fcf6e7,2 năm trước,Chưa bao giờ làm ở công ty kiểu gia đình như v...,Việt Mỹ Ssu,NaN,NaN,Việt Mỹ Ssu (15) outsource 151-500 198 A5 Hoàn...,Chưa bao giờ làm ở công ty kiểu gia đình như v...,Chưa bao giờ làm ở công ty kiểu gia đình như v...


In [27]:
# Xuất file kết quả
data_cleaned.to_csv("reviews_cleaned.csv", index=False, encoding='utf-8-sig')
print("Đã xuất file reviews_cleaned.csv")
print(f"Tổng số reviews: {len(data_cleaned)}")
print(f"Số công ty: {data_cleaned['company_name'].nunique()}")

Đã xuất file reviews_cleaned.csv
Tổng số reviews: 18764
Số công ty: 357


In [ ]:
# Lưu file CSV cho 100 dòng đầu để labeling
data_100 = data_cleaned.head(100).copy()

# Chọn các cột cần thiết
data_100_export = data_100[["reviewer_id", "company_name", "review_content_masked"]].copy()

# Lưu file để chuẩn bị gán nhãn
data_100_export.to_csv("reviews_100_for_labeling.csv", index=False, encoding='utf-8-sig')
print("Đã lưu file reviews_100_for_labeling.csv (100 dòng đầu, sẵn sàng để gán nhãn)")
print(f"\nMột số mẫu:")
data_100_export.head(10)

Đã lưu file Vietnamese_100_cleaned_masked.csv (100 dòng đầu, đã mask tên công ty thành [COMPANY])


In [ ]:
# Tạo thống kê theo công ty
print("=== THỐNG KÊ THEO CÔNG TY ===\n")
company_stats = data_cleaned.groupby('company_name').agg({
    'reviewer_id': 'count',
    'review_content_masked': lambda x: x.str.len().mean()
}).round(0)
company_stats.columns = ['Số reviews', 'Độ dài TB (ký tự)']
company_stats = company_stats.sort_values('Số reviews', ascending=False)

print("Top 20 công ty có nhiều reviews nhất:")
print(company_stats.head(20))

# Lưu thống kê
company_stats.to_csv("company_statistics.csv", encoding='utf-8-sig')
print("\nĐã lưu thống kê vào company_statistics.csv")

Không tìm thấy file Vietnamese_100_cleaned_labeled.csv


In [ ]:
# Phân tích độ dài reviews
import matplotlib.pyplot as plt
import numpy as np

review_lengths = data_cleaned['review_content_masked'].str.len()

print("=== PHÂN TÍCH ĐỘ DÀI REVIEWS ===")
print(f"Độ dài trung bình: {review_lengths.mean():.0f} ký tự")
print(f"Độ dài trung vị: {review_lengths.median():.0f} ký tự")
print(f"Độ dài ngắn nhất: {review_lengths.min():.0f} ký tự")
print(f"Độ dài dài nhất: {review_lengths.max():.0f} ký tự")
print(f"Độ lệch chuẩn: {review_lengths.std():.0f} ký tự")

# Vẽ histogram
plt.figure(figsize=(12, 6))
plt.hist(review_lengths, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Độ dài review (ký tự)')
plt.ylabel('Số lượng')
plt.title('Phân bố độ dài reviews')
plt.axvline(review_lengths.mean(), color='red', linestyle='--', label=f'Trung bình: {review_lengths.mean():.0f}')
plt.axvline(review_lengths.median(), color='green', linestyle='--', label=f'Trung vị: {review_lengths.median():.0f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nPhân vị 25%: {review_lengths.quantile(0.25):.0f} ký tự")
print(f"Phân vị 75%: {review_lengths.quantile(0.75):.0f} ký tự")

Checking for label columns...
Đã tạo file Vietnamese_100_cleaned_labeled_masked.csv với Labels từ rating columns

Số dòng có Labels: 100/100


,Combined_Review,Labels
0,Phúc lợi khá OK còn công việc thì tùy dự án và...,"Salary & benefits, Training & learning, Manage..."
1,"Môi trường năng động, trẻ, chính sách bảo mật ...","Salary & benefits, Training & learning, Manage..."
2,Một nơi tốt để học và phát triểu bản thân. Địn...,"Salary & benefits, Training & learning, Manage..."
3,"Review mang tính góp ý, không có ý định chê cô...","Salary & benefits, Training & learning, Manage..."
4,"Cty trẻ, năng động, lead có tâm, benefit ít. C...","Salary & benefits, Training & learning, Manage..."
5,Khách hàng chủ yếu thị trường Mỹ nên môi trườn...,"Salary & benefits, Training & learning, Manage..."
6,Hợp để fresher lấy kinh nghiệm. Môi trường cởi...,"Salary & benefits, Training & learning, Manage..."
7,Môi trường phát triển tốt. Cơ hội giao tiếp bằ...,"Salary & benefits, Training & learning, Manage..."
8,"Môi trường năng động, phù hợp cho sinh viên mớ...","Salary & benefits, Training & learning, Manage..."
9,Môi trường khá tốt. Mô trường tương đối thân t...,"Salary & benefits, Training & learning, Manage..."


In [ ]:
# Kiểm tra và hiển thị một số review mẫu
print("=== MỘT SỐ REVIEW MẪU SAU KHI XỬ LÝ ===\n")

sample_reviews = data_cleaned.sample(min(5, len(data_cleaned)), random_state=42)

for idx, row in sample_reviews.iterrows():
    print(f"Công ty: {row['company_name']}")
    print(f"Review: {row['review_content_masked'][:200]}...")
    print(f"Độ dài: {len(row['review_content_masked'])} ký tự")
    print("-" * 80)
    print()

Đã gộp 3 cột thành 1 cột 'Combined_Review' cho 19938 dòng đầu

Độ dài trung bình: 385 ký tự


,Combined_Review
0,Phúc lợi khá OK còn công việc thì tùy dự án và...
1,"Môi trường năng động, trẻ, chính sách bảo mật ..."
2,Một nơi tốt để học và phát triểu bản thân. Địn...
3,"Review mang tính góp ý, không có ý định chê cô..."
4,"Cty trẻ, năng động, lead có tâm, benefit ít. C..."
5,Khách hàng chủ yếu thị trường Mỹ nên môi trườn...
6,Hợp để fresher lấy kinh nghiệm. Môi trường cởi...
7,Môi trường phát triển tốt. Cơ hội giao tiếp bằ...
8,"Môi trường năng động, phù hợp cho sinh viên mớ..."
9,Môi trường khá tốt. Mô trường tương đối thân t...


In [ ]:
# Phân tích từ tiếng Anh trong reviews
import re
from collections import Counter
import pandas as pd
import nltk
nltk.download('words', quiet=True)
from nltk.corpus import words

english_vocab = set(words.words())

# Blacklist: từ tiếng Việt không dấu trùng với tiếng Anh
VIETNAMESE_BLACKLIST = {
    'co', 'con', 'ban', 'chi', 'cho', 'chua', 'chu', 'cua', 'da', 'dan',
    'dao', 'den', 'di', 'do', 'doi', 'du', 'duoc', 'giao', 'gio', 'hai',
    'hay', 'het', 'ho', 'hoc', 'khi', 'khong', 'la', 'lam', 'lon', 'ma',
    'mai', 'mat', 'met', 'moi', 'nam', 'nao', 'nay', 'nha', 'nhi', 'nho',
    'nhu', 'nhung', 'noi', 'qua', 'quan', 'rat', 'roi', 'sau', 'se', 'tai',
    'tam', 'than', 'the', 'thi', 'thoi', 'thu', 'tot', 'trang', 'tre', 'tren',
    'trong', 'tu', 'va', 've', 'vi', 'voi', 'xa', 'xin', 'xu', 'ca', 'cac',
    'anh', 'chi', 'em', 'ong', 'ba', 'de', 'dong', 'long', 'tin', 'tri', 'sinh',
}

try:
    import enchant
    d = enchant.Dict("en_US")
    
    def extract_english_words_enchant(text):
        words_found = re.findall(r'\b[a-zA-Z]{4,}\b', str(text).lower())
        return [w for w in words_found if d.check(w) and w not in VIETNAMESE_BLACKLIST]
    
    # Tổng hợp tất cả từ tiếng Anh
    all_words = []
    for review in data_cleaned["review_content_masked"]:
        all_words.extend(extract_english_words_enchant(review))
    
    # Thống kê tần suất
    word_counts = Counter(all_words)
    print(f"\n=== PHÂN TÍCH TỪ TIẾNG ANH ===")
    print(f"Tổng số từ tiếng Anh tìm thấy: {len(word_counts)} từ khác nhau")
    print(f"\nTop 30 từ xuất hiện nhiều nhất:")
    for word, count in word_counts.most_common(30):
        print(f"  {word}: {count}")
        
except ImportError:
    print("Cần cài đặt thư viện enchant để phân tích từ tiếng Anh")
    print("Chạy: pip install pyenchant")

False